# E06 S03 holdout identifiability evaluation

This notebook reads existing S02 neutering-cycle artifacts and asks an independent J2 holdout judge whether the historical period can be inferred from each text condition. Executing the sampling cells calls OpenRouter; validation should only check notebook structure.

In [1]:
from pathlib import Path

S02_RUN_DIR = Path("data/runs/e06_cycle_model_bench/gpt54__qwen397b")
OUT_DIR = Path("data/runs/e06_identifiability_model_bench/gpt54__qwen397b")
RECOMPUTE_FROM_EXISTING = False
RUN_J2_CALLS = True

# For model-benchmark runs, use for example:
# S02_RUN_DIR = Path("data/runs/e06_cycle_model_bench/gpt54__qwen397b")
# OUT_DIR = Path("data/runs/e06_identifiability_model_bench/gpt54__qwen397b")

MODEL_J2 = "anthropic/claude-haiku-4.5"
J2_TEMPERATURE = 1.0
J2_N_SAMPLES = 5

JUDGE_BLIND_THRESHOLD = 0.4

# Recompute aggregates from existing sample files without new J2 calls.
# Default is True (safe): only rescore, do not call OpenRouter.


POINTS = [
    {
        "point_label": "calm_2021",
        "run_date": "2021-10-20",
        "true_year": 2021,
        "true_month": 10,
    },
    {
        "point_label": "shock_war",
        "run_date": "2022-03-25",
        "true_year": 2022,
        "true_month": 3,
    },
]


In [2]:
import json
import os
import re
from collections import Counter
from statistics import mean
from typing import Any, Optional

from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(anchor_path: Path) -> Path:
    if anchor_path.is_absolute():
        return anchor_path.parent.parent.parent
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / anchor_path).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate configured S02_RUN_DIR from cwd={Path.cwd()}: {anchor_path}")


PROJECT_ROOT = find_project_root(S02_RUN_DIR)
load_dotenv(PROJECT_ROOT / ".env")
S02_RUN_DIR = S02_RUN_DIR if S02_RUN_DIR.is_absolute() else PROJECT_ROOT / S02_RUN_DIR
OUT_DIR = OUT_DIR if OUT_DIR.is_absolute() else PROJECT_ROOT / OUT_DIR


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.write("\n")


def resolve_artifact_path(point_dir: Path, text_path: str | Path) -> Path:
    path = Path(text_path)
    if path.is_absolute():
        return path
    if path.exists():
        return path
    project_relative = PROJECT_ROOT / path
    if project_relative.exists():
        return project_relative
    return point_dir / path


def load_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing input text artifact: {path}")
    return path.read_text(encoding="utf-8")


def relative_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def get_final_status(final_record: dict[str, Any]) -> str:
    for key in ("final_status", "status", "outcome", "decision"):
        value = final_record.get(key)
        if value is not None:
            return str(value)
    return "unknown"

In [3]:
J2_IDENTIFIABILITY_SYSTEM = """
You are an independent holdout judge for a blind identifiability evaluation.

Your task is to infer the historical period described by the provided text alone.
The text may be raw or deliberately neutralized to remove identifying markers.

Rules:
- Rely only on the content of the text.
- Do not rely on metadata, filenames, labels, user context, evaluation setup, or any hidden assumptions about why the text was selected.
- Do not assume there is a fixed list of allowed periods.
- If the text is ambiguous, report calibrated uncertainty instead of forcing an answer.
- Return strict JSON only. Do not wrap it in prose.

Return exactly this JSON shape:
{
  "can_identify_period": true,
  "predicted_year": 2021,
  "predicted_month": 10,
  "predicted_period_description": "...",
  "confidence": 0.0,
  "evidence_type": "explicit_marker | macro_configuration | symbolic_statement | absence_pattern | guess | cannot_determine",
  "rationale": "...",
  "identifying_evidence": [
    "..."
  ]
}

Field rules:
- can_identify_period must be a boolean.
- predicted_year must be an integer or null.
- predicted_month must be an integer from 1 to 12 or null.
- confidence must be a float from 0 to 1.
- evidence_type must be exactly one of: explicit_marker, macro_configuration, symbolic_statement, absence_pattern, guess, cannot_determine.
- identifying_evidence must be a list of short evidence phrases from the text.
- If you cannot identify the period, set can_identify_period to false, predicted_year to null, predicted_month to null, confidence to 0.4 or lower, and evidence_type to cannot_determine or guess.
""".strip()


def make_j2_user_prompt(summary_text: str) -> str:
    return (
        "Infer the historical period from the following text alone. "
        "Return strict JSON using the required schema.\n\n"
        "TEXT:\n"
        f"{summary_text}"
    )


def call_j2(system_prompt: str, user_prompt: str, *, model: str, temperature: float) -> str:
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    content = response.choices[0].message.content
    if not content:
        raise ValueError("J2 returned an empty response")
    return content

In [4]:
EVIDENCE_TYPES = {
    "explicit_marker",
    "macro_configuration",
    "symbolic_statement",
    "absence_pattern",
    "guess",
    "cannot_determine",
}

REQUIRED_J2_FIELDS = {
    "can_identify_period",
    "predicted_year",
    "predicted_month",
    "predicted_period_description",
    "confidence",
    "evidence_type",
    "rationale",
    "identifying_evidence",
}

PERIOD_WEIGHTS = {
    "exact_month": 1.0,
    "adjacent_month": 0.75,
    "same_quarter": 0.5,
    "same_year": 0.25,
    "wrong_period": 0.0,
    "no_prediction": 0.0,
}


def extract_json_response(raw_response: str) -> dict[str, Any]:
    text = raw_response.strip()
    fence_match = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Could not parse J2 response as JSON: {exc}") from exc

    if not isinstance(parsed, dict):
        raise ValueError("J2 JSON response must be an object")
    return parsed


def validate_j2_parsed(parsed: dict[str, Any]) -> None:
    missing = sorted(REQUIRED_J2_FIELDS - set(parsed))
    if missing:
        raise ValueError(f"J2 JSON response is missing fields: {missing}")

    if not isinstance(parsed["can_identify_period"], bool):
        raise TypeError("can_identify_period must be a boolean")
    if parsed["predicted_year"] is not None and not isinstance(parsed["predicted_year"], int):
        raise TypeError("predicted_year must be an integer or null")
    if parsed["predicted_month"] is not None:
        if not isinstance(parsed["predicted_month"], int) or not 1 <= parsed["predicted_month"] <= 12:
            raise ValueError("predicted_month must be an integer from 1 to 12 or null")
    if not isinstance(parsed["confidence"], (int, float)):
        raise TypeError("confidence must be numeric")
    if parsed["evidence_type"] not in EVIDENCE_TYPES:
        raise ValueError(f"Invalid evidence_type: {parsed['evidence_type']}")
    if not isinstance(parsed["identifying_evidence"], list):
        raise TypeError("identifying_evidence must be a list")


def clamp_confidence(value: Any) -> float:
    confidence = float(value)
    return max(0.0, min(1.0, confidence))


def month_index(year: int, month: int) -> int:
    return year * 12 + month


def month_distance(
    pred_year: Optional[int],
    pred_month: Optional[int],
    true_year: int,
    true_month: int,
) -> Optional[int]:
    if pred_year is None or pred_month is None:
        return None
    if not (1 <= int(pred_month) <= 12):
        return None
    return abs(month_index(int(pred_year), int(pred_month)) - month_index(int(true_year), int(true_month)))


def same_calendar_quarter(year_a: int, month_a: int, year_b: int, month_b: int) -> bool:
    if year_a != year_b:
        return False
    return (month_a - 1) // 3 == (month_b - 1) // 3


def score_identification(parsed: dict, true_year: int, true_month: int) -> dict:
    confidence = clamp_confidence(parsed.get("confidence", 0.0))
    can_identify = parsed.get("can_identify_period") is True
    pred_year = parsed.get("predicted_year")
    pred_month = parsed.get("predicted_month")

    dist = month_distance(pred_year, pred_month, true_year, true_month) if can_identify else None

    exact_success = bool(
        can_identify
        and pred_year == true_year
        and pred_month == true_month
    )
    adjacent_month_success = bool(
        can_identify
        and dist is not None
        and dist <= 1
    )
    same_quarter_success = bool(
        can_identify
        and pred_year is not None
        and pred_month is not None
        and same_calendar_quarter(int(pred_year), int(pred_month), true_year, true_month)
    )
    same_year_success = bool(
        can_identify
        and pred_year == true_year
    )

    if not can_identify or pred_year is None or pred_month is None:
        period_success_level = "no_prediction"
    elif exact_success:
        period_success_level = "exact_month"
    elif adjacent_month_success:
        period_success_level = "adjacent_month"
    elif same_quarter_success:
        period_success_level = "same_quarter"
    elif same_year_success:
        period_success_level = "same_year"
    else:
        period_success_level = "wrong_period"

    period_weight = PERIOD_WEIGHTS[period_success_level]

    return {
        "true_year": true_year,
        "true_month": true_month,
        "predicted_year": pred_year,
        "predicted_month": pred_month,
        "confidence": confidence,
        "month_distance": dist,
        "exact_success": exact_success,
        "adjacent_month_success": adjacent_month_success,
        "same_quarter_success": same_quarter_success,
        "same_year_success": same_year_success,
        "period_success_level": period_success_level,
        "period_weight": period_weight,
        "exact_weighted_success": confidence if exact_success else 0.0,
        "period_weighted_success": round(confidence * period_weight, 6),
        "scoring_version": "period_aware_v2",
        # backward compat
        "success": exact_success,
        "weighted_success": confidence if exact_success else 0.0,
    }


In [5]:
def build_conditions_for_point(
    point: dict[str, Any], final_record: dict[str, Any], point_dir: Path
) -> list[dict[str, Any]]:
    raw_path = point_dir / "iter_00_raw.md"
    final_iteration = final_record.get("final_iteration")
    final_status = get_final_status(final_record)
    accepted_neutered_exists = isinstance(final_iteration, int) and final_iteration > 0

    conditions = [
        {
            "condition": "raw",
            "text_path": raw_path,
            "accepted_neutered_exists": accepted_neutered_exists,
            "diagnostic": False,
            "final_status": final_status,
        }
    ]

    if accepted_neutered_exists:
        final_candidate_path = final_record.get("final_candidate_path")
        if not final_candidate_path:
            raise KeyError(
                f"{point['point_label']} final_record.json is missing final_candidate_path"
            )
        conditions.append(
            {
                "condition": "final_neutered",
                "text_path": resolve_artifact_path(point_dir, final_candidate_path),
                "accepted_neutered_exists": True,
                "diagnostic": False,
                "final_status": final_status,
            }
        )
    else:
        failed_candidate_path = point_dir / "iter_01_candidate.md"
        if failed_candidate_path.exists():
            conditions.append(
                {
                    "condition": "failed_iter_01_diagnostic",
                    "text_path": failed_candidate_path,
                    "accepted_neutered_exists": False,
                    "diagnostic": True,
                    "final_status": final_status,
                }
            )

    return conditions


def aggregate_samples(
    *,
    point_label: str,
    condition: dict[str, Any],
    samples: list[dict[str, Any]],
    final_status: str,
    input_text_path: Path,
) -> dict[str, Any]:
    scoring = [sample["scoring"] for sample in samples]
    evidence_type_counts = Counter(sample["parsed"].get("evidence_type") for sample in samples)
    period_success_level_counts = Counter(
        row.get("period_success_level", "no_prediction") for row in scoring
    )
    n_samples = len(samples)
    if n_samples == 0:
        raise ValueError("Cannot aggregate zero samples")

    exact_id_rate = mean(1.0 if row.get("exact_success", row.get("success", False)) else 0.0 for row in scoring)
    exact_id_score = mean(row.get("exact_weighted_success", row.get("weighted_success", 0.0)) for row in scoring)
    period_id_score = mean(row.get("period_weighted_success", row.get("weighted_success", 0.0)) for row in scoring)
    adjacent_month_rate = mean(1.0 if row.get("adjacent_month_success", False) else 0.0 for row in scoring)
    same_quarter_rate = mean(1.0 if row.get("same_quarter_success", False) else 0.0 for row in scoring)
    same_year_rate = mean(1.0 if row.get("same_year_success", False) else 0.0 for row in scoring)
    mean_confidence_val = mean(row["confidence"] for row in scoring)

    return {
        "point_label": point_label,
        "condition": condition["condition"],
        "n_samples": n_samples,
        # backward compat exact metrics
        "id_rate": exact_id_rate,
        "id_score": exact_id_score,
        "exact_id_rate": exact_id_rate,
        "exact_id_score": exact_id_score,
        # period-aware metrics
        "adjacent_month_rate": adjacent_month_rate,
        "same_quarter_rate": same_quarter_rate,
        "same_year_rate": same_year_rate,
        "period_id_score": period_id_score,
        "mean_confidence": mean_confidence_val,
        "period_success_level_counts": dict(sorted(period_success_level_counts.items())),
        "evidence_type_counts": dict(sorted(evidence_type_counts.items())),
        "accepted_neutered_exists": condition["accepted_neutered_exists"],
        "diagnostic": condition["diagnostic"],
        "final_status": final_status,
        "input_text_path": relative_path(input_text_path),
    }


def load_existing_condition_samples(
    *,
    point_label: str,
    condition_name: str,
) -> list[dict[str, Any]]:
    condition_out_dir = OUT_DIR / point_label / condition_name
    samples = []
    sample_index = 1
    while True:
        path = condition_out_dir / f"sample_{sample_index:02d}.json"
        if not path.exists():
            break
        samples.append(read_json(path))
        sample_index += 1
    if not samples:
        raise FileNotFoundError(
            f"No sample files found in {condition_out_dir}. "
            "Run with RUN_J2_CALLS=True first to generate samples."
        )
    return samples


def run_condition_samples(
    *,
    point: dict[str, Any],
    condition: dict[str, Any],
    final_status: str,
    point_dir: Path,
) -> dict[str, Any]:
    point_label = point["point_label"]
    condition_name = condition["condition"]
    input_text_path = resolve_artifact_path(point_dir, condition["text_path"])
    condition_out_dir = OUT_DIR / point_label / condition_name

    if RECOMPUTE_FROM_EXISTING:
        raw_samples = load_existing_condition_samples(
            point_label=point_label, condition_name=condition_name
        )
        samples = []
        for sample in raw_samples:
            rescored = score_identification(
                sample["parsed"], point["true_year"], point["true_month"]
            )
            updated_sample = {**sample, "scoring": rescored}
            write_json(
                condition_out_dir / f"sample_{sample['sample_index']:02d}.json",
                updated_sample,
            )
            samples.append(updated_sample)
    elif RUN_J2_CALLS:
        summary_text = load_text(input_text_path)
        samples = []
        for sample_index in range(1, J2_N_SAMPLES + 1):
            raw_response = call_j2(
                J2_IDENTIFIABILITY_SYSTEM,
                make_j2_user_prompt(summary_text),
                model=MODEL_J2,
                temperature=J2_TEMPERATURE,
            )
            parsed = extract_json_response(raw_response)
            validate_j2_parsed(parsed)
            scoring = score_identification(parsed, point["true_year"], point["true_month"])
            sample_payload = {
                "point_label": point_label,
                "condition": condition_name,
                "sample_index": sample_index,
                "model": MODEL_J2,
                "temperature": J2_TEMPERATURE,
                "input_text_path": relative_path(input_text_path),
                "raw_response": raw_response,
                "parsed": parsed,
                "scoring": scoring,
            }
            write_json(condition_out_dir / f"sample_{sample_index:02d}.json", sample_payload)
            samples.append(sample_payload)
    else:
        raise RuntimeError(
            "Neither RECOMPUTE_FROM_EXISTING nor RUN_J2_CALLS is True. "
            "Set one of them to True to proceed."
        )

    aggregate = aggregate_samples(
        point_label=point_label,
        condition=condition,
        samples=samples,
        final_status=final_status,
        input_text_path=input_text_path,
    )
    write_json(condition_out_dir / "aggregate.json", aggregate)
    return aggregate


In [6]:
def run_point(point: dict[str, Any]) -> dict[str, Any]:
    point_label = point["point_label"]
    point_dir = S02_RUN_DIR / point_label
    final_record_path = point_dir / "final_record.json"
    if not final_record_path.exists():
        raise FileNotFoundError(f"Missing S02 final_record.json: {final_record_path}")

    final_record = read_json(final_record_path)
    final_status = get_final_status(final_record)
    final_iteration = final_record.get("final_iteration")
    accepted_neutered_exists = isinstance(final_iteration, int) and final_iteration > 0
    conditions = build_conditions_for_point(point, final_record, point_dir)
    aggregates = [
        run_condition_samples(
            point=point,
            condition=condition,
            final_status=final_status,
            point_dir=point_dir,
        )
        for condition in conditions
    ]

    raw_aggregate = next(row for row in aggregates if row["condition"] == "raw")
    # Use period_id_score for judge_blind: adjacent-month prediction is not blind
    raw_period_id_score = raw_aggregate["period_id_score"]
    judge_blind = raw_period_id_score < JUDGE_BLIND_THRESHOLD

    for aggregate in aggregates:
        aggregate["id_delta_vs_raw"] = raw_aggregate["exact_id_score"] - aggregate["exact_id_score"]
        aggregate["id_rate_delta_vs_raw"] = raw_aggregate["exact_id_rate"] - aggregate["exact_id_rate"]
        aggregate["period_delta_vs_raw"] = raw_period_id_score - aggregate["period_id_score"]
        aggregate["period_rate_delta_vs_raw"] = (
            raw_aggregate["adjacent_month_rate"] - aggregate["adjacent_month_rate"]
        )
        aggregate["judge_blind"] = judge_blind
        write_json(OUT_DIR / point_label / aggregate["condition"] / "aggregate.json", aggregate)

    point_summary = {
        "point_label": point_label,
        "run_date": point["run_date"],
        "true_year": point["true_year"],
        "true_month": point["true_month"],
        "accepted_neutered_exists": accepted_neutered_exists,
        "judge_blind": judge_blind,
        "raw_id_score": raw_aggregate["exact_id_score"],
        "raw_id_rate": raw_aggregate["exact_id_rate"],
        "raw_period_id_score": raw_period_id_score,
        "condition_aggregates": aggregates,
    }
    if not accepted_neutered_exists:
        point_summary["note"] = (
            "No accepted neutered candidate exists for this point in the selected run; "
            "failed_iter_01_diagnostic is evaluated only to inspect the trade-off "
            "between identifiability and signal collapse."
        )

    write_json(OUT_DIR / point_label / "point_summary.json", point_summary)
    return point_summary


point_summaries = [run_point(point) for point in POINTS]
condition_aggregates = [
    aggregate
    for point_summary in point_summaries
    for aggregate in point_summary["condition_aggregates"]
]

comparison_columns = [
    "point_label",
    "condition",
    "n_samples",
    "exact_id_rate",
    "exact_id_score",
    "period_id_score",
    "mean_confidence",
    "id_delta_vs_raw",
    "period_delta_vs_raw",
    "adjacent_month_rate",
    "same_quarter_rate",
    "same_year_rate",
    "judge_blind",
    "final_status",
    "accepted_neutered_exists",
    "diagnostic",
]
comparison_rows = [
    {key: aggregate.get(key) for key in comparison_columns}
    for aggregate in condition_aggregates
]

s03_summary = {
    "scoring_version": "period_aware_v2",
    "model_config": {
        "model_j2": MODEL_J2,
        "openrouter_base_url": "https://openrouter.ai/api/v1",
    },
    "sampling_config": {
        "temperature": J2_TEMPERATURE,
        "n_samples": J2_N_SAMPLES,
        "judge_blind_threshold": JUDGE_BLIND_THRESHOLD,
    },
    "recompute_config": {
        "recompute_from_existing": RECOMPUTE_FROM_EXISTING,
        "run_j2_calls": RUN_J2_CALLS,
    },
    "condition_aggregates": condition_aggregates,
    "point_summaries": point_summaries,
    "final_comparison_rows": comparison_rows,
}
write_json(OUT_DIR / "s03_summary.json", s03_summary)


In [7]:
if "comparison_columns" not in globals():
    comparison_columns = [
        "point_label",
        "condition",
        "n_samples",
        "exact_id_rate",
        "exact_id_score",
        "period_id_score",
        "mean_confidence",
        "id_delta_vs_raw",
        "period_delta_vs_raw",
        "adjacent_month_rate",
        "same_quarter_rate",
        "same_year_rate",
        "judge_blind",
        "final_status",
        "accepted_neutered_exists",
        "diagnostic",
    ]

if "comparison_rows" not in globals():
    with (OUT_DIR / "s03_summary.json").open("r", encoding="utf-8") as f:
        s03_summary = json.load(f)
    comparison_rows = s03_summary["final_comparison_rows"]

try:
    import pandas as pd

    display(pd.DataFrame(comparison_rows)[comparison_columns])
except ImportError:
    print(json.dumps(comparison_rows, ensure_ascii=False, indent=2))


,point_label,condition,n_samples,exact_id_rate,exact_id_score,period_id_score,mean_confidence,id_delta_vs_raw,period_delta_vs_raw,adjacent_month_rate,same_quarter_rate,same_year_rate,judge_blind,final_status,accepted_neutered_exists,diagnostic
0,calm_2021,raw,5,0.0,0.00,0.69,0.920,0.00,0.00,1.0,0.0,1.0,False,hard_capped,True,False
1,calm_2021,final_neutered,5,0.0,0.00,0.00,0.762,0.00,0.69,0.0,0.0,0.0,False,hard_capped,True,False
2,shock_war,raw,5,1.0,0.95,0.95,0.950,0.00,0.00,1.0,1.0,1.0,False,signal_collapse,True,False
3,shock_war,final_neutered,5,1.0,0.92,0.92,0.920,0.03,0.03,1.0,1.0,1.0,False,signal_collapse,True,False


## Scoring note

Exact month scoring is retained, but the main identifiability metric is now period-aware.

This is necessary because the project unit is `run_date` (survey publication date), while summaries may contain news and economic data from adjacent months. A judge predicting the adjacent month with high confidence is not blind; it still identified the historical period.

Period weights:
- `exact_month` → 1.0
- `adjacent_month` → 0.75 (within 1 calendar month of `run_date`)
- `same_quarter` → 0.5
- `same_year` → 0.25
- `wrong_period` or `no_prediction` → 0.0

`judge_blind` is now derived from `raw_period_id_score < JUDGE_BLIND_THRESHOLD`, not from exact scoring.
This fixes the `calm_2021` false negative: J2 predicts September 2021 (adjacent month) with high confidence,
which means it identified the historical episode and is not blind.
